## एम्बेडिंग्ज

आपल्या मागील उदाहरणात, आपण उच्च-आयामी बॅग-ऑफ-वर्ड्स व्हेक्टरवर कृत्य केले जे लांबीचे होते `vocab_size`, आणि आपण स्पष्टपणे कमी-आयामी स्थानिक प्रतिनिधित्व व्हेक्टरपासून विरळ एक-हॉट प्रतिनिधित्वात रूपांतर करत होतो. हे एक-हॉट प्रतिनिधित्व स्मृतीसक्षम नाही, शिवाय प्रत्येक शब्द स्वतंत्ररित्या विचारला जातो, म्हणजेच एक-हॉट एन्कोडेड व्हेक्टर शब्दांमधील कोणतीही सैद्धांतिक साम्य दर्शवत नाहीत.

या युनिटमध्ये, आपण **News AG** डेटासेटचा अन्वेषण सुरू ठेवू. सुरुवात करण्यासाठी, चला डेटा लोड करू आणि मागील नोटबुकमधून काही व्याख्या मिळवू.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## एम्बेडिंग म्हणजे काय?

**एम्बेडिंग** ची संकल्पना म्हणजे शब्दांना कमी-आयामी घन वेक्टरद्वारे प्रतिनिधित्व करणे, जे कसे तरी त्या शब्दाचा अर्थसंबंध दर्शवते. आपण नंतर अर्थपूर्ण शब्द एम्बेडिंग कसे तयार करायचे यावर चर्चा करू, पण सध्या फक्त एम्बेडिंगला शब्द वेक्टरची आयाम कमी करण्याचा मार्ग म्हणून समजून घेऊया.

त्यामुळे, एम्बेडिंग लेयर एका शब्दाला इनपुट म्हणून घेईल आणि निर्दिष्ट केलेल्या `embedding_size` चा आउटपुट वेक्टर तयार करेल. एका अर्थाने, हा `Linear` लेयरसारखा आहे, पण त्याऐवजी जो एक-हॉट एन्कोडेड वेक्टर घेतो, हा शब्द क्रमांक इनपुट म्हणून घेऊ शकेल.

आमच्या नेटवर्कमध्ये एम्बेडिंग लेयरला पहिले लेयर म्हणून वापरून, आपण बॅग-ऑफ-वर्ड्समधून **एम्बेडिंग बॅग** मॉडेलकडे जाऊ शकतो, जिथे आपण प्रथम आपल्या मजकुरातील प्रत्येक शब्दाला त्याच्या संबंधित एम्बेडिंगमध्ये रूपांतरित करतो आणि नंतर त्या सर्व एम्बेडिंग्सवर काही एकत्रिकरण फंक्शन काढतो, जसे की `sum`, `average` किंवा `max`.

![पाच अनुक्रम शब्दांसाठी एम्बेडिंग वर्गाकार दाखवणारी प्रतिमा.](../../../../../translated_images/mr/embedding-classifier-example.b77f021a7ee67eee.webp)

आमची वर्गीकरणीय न्यूरल नेटवर्क एम्बेडिंग लेयरने सुरू होईल, नंतर एकत्रिकरण लेयर आणि त्यावर एक रेषीय वर्गीकारक:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### बदलत्या अनुक्रमणिका आकाराचा सामना करणे

या आर्किटेक्चरच्या परिणामी, आमच्या नेटवर्कसाठी मिनीबॅचेस विशिष्ट पद्धतीने तयार कराव्या लागतील. मागील युनिटमध्ये, जेव्हा बॅग-ऑफ-वर्ड्स वापरत होतो, तेव्हा मिनीबॅचमधील सर्व BoW टेन्सरची आकार समान होती `vocab_size`, आमच्या मजकूराच्या अनुक्रमाची वास्तविक लांबी काय आहे याची पर्वा न करता. एकदा आपण शब्द एम्बेडिंगकडे गेला, तेव्हा प्रत्येक मजकूर नमुन्यामध्ये शब्दांची संख्या बदलत्या स्वरूपात असेल, आणि जेव्हा आपण त्या नमुन्यांना मिनीबॅचेसमध्ये एकत्रित करू, तेव्हा आपल्याला काही पॅडिंग लागू करावी लागेल.

हे datasource ला `collate_fn` फंक्शन प्रदान करण्याच्या त्याच तंत्राचा वापर करून करता येऊ शकते:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### एम्बेडिंग क्लासिफायर प्रशिक्षण

आता जेव्हा आपण योग्य डेटालोडर निश्चित केला आहे, तेव्हा आपण मागील युनिटमध्ये परिभाषित केलेले प्रशिक्षण कार्य वापरून मॉडेल प्रशिक्षण करू शकतो:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **टीप**: आपण येथे केवळ 25k रेकॉर्डसाठी प्रशिक्षण देत आहोत (पूर्ण epoch च्या तुलनेत कमी), वेळेच्या दृष्टीने, परंतु आपण प्रशिक्षण चालू ठेवू शकता, अनेक epochs साठी प्रशिक्षण देण्यासाठी एक फंक्शन लिहू शकता, आणि उच्च अचूकता साध्य करण्यासाठी शिक्षण दराच्या पॅरामीटरसह प्रयोग करू शकता. आपण सुमारे 90% अचूकतेपर्यंत पोहोचू शकता.


### EmbeddingBag लेयर आणि परिवर्तनीय-लांबीच्या अनुक्रमाचे प्रतिनिनिधित्व

मागील आर्किटेक्चरमध्ये, आम्हाला सर्व अनुक्रम समान लांबीपर्यंत पॅड करावे लागले जेणेकरून ते मिनिबॅचमध्ये बसतील. ही परिवर्तनीय लांबीच्या अनुक्रमांचे प्रतिनिधित्व करण्याची सर्वात कार्यक्षम पद्धत नाही - आणखी एक पद्धत म्हणजे **offset** वेक्टर वापरणे, जे एका मोठ्या वेक्टरमध्ये साठवलेल्या सर्व अनुक्रमांच्या ऑफसेटस धरते.

![ऑफसेट अनुक्रम प्रतिनिधित्व दाखवणारी प्रतिमा](../../../../../translated_images/mr/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **सूचना**: वर दिलेल्या चित्रावर, आम्ही अक्षरांचा अनुक्रम दाखविला आहे, परंतु आमच्या उदाहरणात आपण शब्दांच्या अनुक्रमांवर काम करत आहोत. तरीही, ऑफसेट वेक्टरसह अनुक्रमांचे प्रतिनिधित्व करण्याचा सामान्य तत्त्व समान राहतो.

ऑफसेट प्रतिनिधित्वावर काम करण्यासाठी, आम्ही [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) लेयर वापरतो. हे `Embedding` सारखेच आहे, परंतु हे कंटेंट वेक्टर आणि ऑफसेट वेक्टर इनपुट म्हणून घेतो, आणि त्यात averaging लेयर देखील समाविष्ट आहे, जो `mean`, `sum` किंवा `max` असू शकतो.

येथे एक सुधारित नेटवर्क आहे जे `EmbeddingBag` वापरते:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

प्रशिक्षणासाठी डेटासेट तयार करण्यासाठी, आपल्याला एक रूपांतरण फंक्शन प्रदान करणे आवश्यक आहे जे ऑफसेट व्हेक्टर तयार करेल:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

लक्षात घ्या, की पूर्वीच्या सर्व उदाहरणांपासून वेगळे, आता आमच्या नेटवर्कला दोन पॅरामीटर्स स्वीकारावे लागतात: डेटा व्हेक्टर आणि ऑफसेट व्हेक्टर, जे वेगवेगळ्या आकारांचे आहेत. तसेच, आमचा डेटा लोडर देखील 2 ऐवजी 3 मूल्ये प्रदान करतो: दोन्ही टेक्स्ट आणि ऑफसेट व्हेक्टर फीचर्स म्हणून दिले जातात. त्यामुळे, आम्हाला आमच्या प्रशिक्षण फंक्शनमध्ये थोडे बदल करण्याची गरज आहे जेणेकरून त्यांची योग्य काळजी घेतली जाईल:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## अर्थपूर्ण समाकलने: Word2Vec

आपल्या मागील उदाहरणात, मॉडेल समाकलन स्तराने शब्दांना वेक्टर प्रतिनिधित्वाकडे नकाशित करणे शिकलं, मात्र, या प्रतिनिधित्वाला फारसा अर्थपूर्ण अर्थ नव्हता. असे वेक्टर प्रतिनिधित्व शिकणे चांगले होईल, ज्यात समानार्थी किंवा तत्सम शब्द अशा वेक्टरशी संबंधित असतील जे काही विशिष्ट वेक्टर अंतर (उदा. युक्लिडीयन अंतर)च्या दृष्टीने एकमेकांच्या जवळ असतील.

तसे करायचे असल्यास, आपल्याला आपल्या समाकलन मॉडेलला विशिष्ट पद्धतीने मोठ्या प्रमाणावर मजकुरावर पूर्व-प्रशिक्षित करणे आवश्यक आहे. अर्थपूर्ण समाकलन शिकण्याच्या सुरुवातीच्या पद्धतींपैकी एक [Word2Vec](https://en.wikipedia.org/wiki/Word2vec) आहे. हे दोन मुख्य रचनांवर आधारित आहे जे शब्दांचे वितरीत प्रतिनिधित्व तयार करण्यासाठी वापरल्या जातात:

 - **सतत शब्दांच्या थैची पिशवी** (CBoW) — या रचनेत, आपण मॉडेलला आजूबाजूच्या संदर्भातून एखादा शब्द भाकीत करण्यासाठी प्रशिक्षण देतो. दिलेला n-gram $(W_{-2},W_{-1},W_0,W_1,W_2)$, मॉडेलचे लक्ष्य आहे $(W_{-2},W_{-1},W_1,W_2)$ पासून $W_0$ भाकीत करणे.
 - **सतत स्किप-ग्राम** हे CBoW च्या उलट आहे. मॉडेल वर्तमान शब्द भाकीत करण्यासाठी आजूबाजूच्या संदर्भ शब्दांच्या विंडोचा वापर करते.

CBoW वेगवान आहे, तर स्किप-ग्राम हळू आहे, पण दुर्मिळ शब्दांचे चांगले प्रतिनिधित्व करते.

![शब्दांना वेक्टर्समध्ये रूपांतरित करण्यासाठी CBoW आणि स्किप-ग्राम अल्गोरिदम दर्शविणारे छायाचित्र.](../../../../../translated_images/mr/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Google News डेटासेटवर पूर्व-प्रशिक्षित word2vec समाकलनासह प्रयोग करण्यासाठी, आपण **gensim** लायब्ररी वापरू शकतो. खाली 'neural' शी सर्वात साम्य असलेले शब्द आढळतात

> **टीप:** जेव्हा तुम्ही प्रथम वेक्टर तयार करता, तेव्हा त्यांना डाउनलोड करण्यात काही वेळ लागू शकतो!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


आपण शब्दातून व्हेक्टर एम्बेडिंग देखील गणना करू शकतो, जे वर्गीकरण मॉडेलच्या प्रशिक्षणासाठी वापरले जातील (स्पष्टतेसाठी, आम्ही फक्त व्हेक्टरचे पहिले 20 घटक दाखवितो):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

सेमॅन्टिकल एम्बेडिंग्जबद्दल छान गोष्ट म्हणजे आपण व्हेक्टर एनकोडिंगमध्ये बदल करू शकतो ज्यामुळे अर्थ बदलेल. उदाहरणार्थ, आपण असा शब्द शोधू शकतो, ज्याचे व्हेक्टर प्रतिनिधित्व शब्द *king* आणि *woman* च्या जवळपास असेल, आणि शब्द *man* पासून जितके शक्य तितके दूर असेल:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

दोन्ही CBoW आणि Skip-Grams हे "भविष्यवाणी करणारी" एम्बेडिंग्ज आहेत, कारण ते फक्त स्थानिक संदर्भांचा विचार करतात. Word2Vec जागतिक संदर्भाचा फायदा घेत नाही.

**FastText**, Word2Vec वर आधारित असून प्रत्येक शब्दासाठी आणि प्रत्येक शब्दात सापडणाऱ्या कॅरेक्टर n-ग्रामसाठी व्हेक्टर प्रतिनिधित्व शिकवते. प्रतिनिधित्वांच्या मूल्यांची सरासरी प्रत्येक प्रशिक्षण टप्यात एका व्हेक्टरमध्ये केली जाते. यामुळे पूर्व-प्रशिक्षणात बरेच अतिरिक्त गणना होते पण हे शब्द एम्बेडिंग्जना उप-शब्द माहिती एन्कोड करण्यास सक्षम करते.

आणखी एक पद्धत, **GloVe**, सह-संवेदन मॅट्रिक्सच्या संकल्पनेचा वापर करते, आणि न्यूरल पद्धती वापरून सह-संवेदन मॅट्रिक्सला अधिक अभिव्यक्तीक्षम आणि असरेखीय शब्द व्हेक्टरमध्ये विघटित करते.

तुम्ही उदाहरणात embeddings FastText आणि GloVe मध्ये बदलून खेळू शकता, कारण gensim विविध शब्द एम्बेडिंग मॉडेल्सला समर्थन देतो.


## PyTorch मध्ये पूर्व-प्रशिक्षित एम्बेडिंग वापरणे

आपण वर दिलेला उदाहरण बदलून आपल्या एम्बेडिंग स्तरातील मॅट्रिक्समध्ये सेमांटिक एम्बेडिंगसह, जसे की Word2Vec, पूर्व-भरली जाऊ शकते. आम्हाला लक्षात घ्यावे लागेल की पूर्व-प्रशिक्षित एम्बेडिंग आणि आपल्या मजकूर संग्रहाचे शब्दसंग्रह बहुधा जुळणार नाहीत, त्यामुळे आपण दुर्लक्षित शब्दांसाठी वजनांना यादृच्छिक मूल्यांनी प्रारंभ करू:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


चला आता आपला मॉडेल प्रशिक्षित करूया. लक्षात ठेवा की मॉडेल प्रशिक्षणासाठी लागणारा वेळ मागील उदाहरणाच्या तुलनेत खूप जास्त आहे, कारण एम्बेडिंग लेयरचा आकार मोठा आहे, आणि त्यामुळे खूप जास्त संख्या असलेल्या पॅरामिटर्स आहेत. तसेच, यामुळे जर आपल्याला ओव्हरफिटिंग टाळायचे असेल तर आपल्याला आपल्या मॉडेलला अधिक उदाहरणांवर प्रशिक्षण देणे आवश्यक असू शकते.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

आमच्या बाबतीत आम्हाला अचूकतेत मोठी वाढ दिसत नाही, जी बहुधा वेगवेगळ्या शब्दसंग्रहामुळे आहे. 
वेगवेगळ्या शब्दसंग्रहाच्या समस्येवर मात करण्यासाठी, आपण खालीलपैकी एक उपाय वापरू शकतो:
* आमच्या शब्दसंग्रहावर word2vec मॉडेल पुनःप्रशिक्षित करा
* आधीच प्रशिक्षित word2vec मॉडेलमधील शब्दसंग्रहासह आमचा डेटासेट लोड करा. डेटासेट लोड करताना वापरलेला शब्दसंग्रह निर्दिष्ट केला जाऊ शकतो.

नंतरचा दृष्टिकोन सोपा वाटतो, विशेषतः कारण PyTorch `torchtext` फ्रेमवर्कमध्ये एंबेडिंगसाठी अंतर्निर्मित समर्थन आहे. उदाहरणार्थ, आपण खालीलप्रमाणे GloVe-आधारित शब्दसंग्रह तयार करू शकतो:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary has the following basic operations:
* `vocab.stoi` dictionary allows us to convert word into its dictionary index
* `vocab.itos` does the opposite - converts number into word
* `vocab.vectors` is the array of embedding vectors, so to get the embedding of a word `s` we need to use `vocab.vectors[vocab.stoi[s]]`

Here is the example of manipulating embeddings to demonstrate the equation **kind-man+woman = queen** (I had to tweak the coefficient a bit to make it work):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

त्या एम्बेडिंगचा वापर करून वर्गीकर्त्याचे प्रशिक्षण देण्यासाठी, प्रथम आपल्याला GloVe शब्दसंग्रह वापरून आपला डेटा संच एन्कोड करावा लागेल:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

वरीलप्रमाणे, सर्व वेक्टर एम्बेडिंग्ज `vocab.vectors` मॅट्रिक्समध्ये संग्रहित आहेत. ही पद्धत एम्बेडिंग लेयरच्या वजनांमध्ये सहज कॉपी करून वज़न लोड करणे खूप सोपे करते:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

आता आपण आपला मॉडेल प्रशिक्षित करू आणि पाहू की आपल्याला उत्तम परिणाम मिळतात का:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

अचूकतेत लक्षणीय वाढ दिसत नाही याचे एक कारण म्हणजे आपल्या डेटासेटमधील काही शब्द पूर्व-प्रशिक्षित GloVe शब्दसंग्रहात नुसतेच गायब आहेत, आणि म्हणूनच ते प्रत्यक्षात दुर्लक्षित केले जातात. या समस्येवर मात करण्यासाठी, आपण आपल्या डेटासेटवर स्वतःचे एम्बेडिंग्ज प्रशिक्षण देऊ शकतो.


## संदर्भात्मक एम्बेडिंग्ज

पारंपरिक प्रीट्रेन्ड एम्बेडिंग सादरीकरणांच्या, जसे की Word2Vec, मुख्य मर्यादा म्हणजे शब्दाच्या अर्थ पारिभाषिक समज दरम्यानचा प्रश्न. जरी प्रीट्रेन्ड एम्बेडिंग्ज संदर्भातील काही शब्दांचा अर्थ पकडू शकतात, तरी प्रत्येक संभाव्य अर्थ एकाच एम्बेडिंगमध्ये कोडित केला जातो. यामुळे खालील मॉडल्समध्ये समस्या येऊ शकते, कारण अनेक शब्द जसे की 'play' वापरल्या जाणाऱ्या संदर्भानुसार वेगवेगळे अर्थ धरतात.

उदाहरणार्थ, दोन वाक्यांमध्ये 'play' या शब्दाचा वेगळा अर्थ असू शकतो:
- मी थिएटरमध्ये एक **play** पाहायला गेलो.
- जॉन त्याच्या मित्रांसोबत **play** करू इच्छितो.

वरील प्रीट्रेन्ड एम्बेडिंग्ज या शब्द 'play' च्या दोन्ही अर्थांचा एकाच एम्बेडिंगमध्ये प्रतिनिधीत्व करतात. ही सीमा वाढवण्यासाठी, आपण **language model** वर आधारित एम्बेडिंग्ज तयार करणे आवश्यक आहे, जे मोठ्या प्रमाणात मजकूरावर प्रशिक्षण घेतलेले असून, शब्द कसे वेगवेगळ्या संदर्भात एकत्र येऊ शकतात हे *माहित आहे*. संदर्भात्मक एम्बेडिंग्जवर चर्चा या ट्यूटोरियलचा विषय नाही, मात्र पुढील युनिटमध्ये भाषा मॉडेल्सचा विचार करताना आपण त्याकडे परत येऊ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
